# Malware hash threat hunt

Join **PE import features** (main dataset) with **MalwareBazaar-style threat intelligence** (lookup) on **MD5 hash**, then interpret families and suspicious API usage.

**Data plane:** all CSV ingestion runs in **Cribl Search** (Generic HTTP dataset provider, `externaldata`, KQL `join`) — not via Pyodide `fetch` and **not** through `config/proxies.yml`.

## Scenario

1. **Main** — PE binary import flags keyed by `hash` (MD5), loaded through a **Generic HTTP dataset provider** + federated dataset.
2. **Enrichment** — TI labels (`signature`, `malware_family`, `tags`) in a **Search lookup**, built from `externaldata` on a companion CSV.
3. **Hunt** — KQL **inner join** on normalized MD5; summarize and chart hits.

Curated samples live in this repo under `public/data/malware-hunt/` (derived from [MalwareBazaar](https://bazaar.abuse.ch/) + [Kaggle PE imports](https://www.kaggle.com/datasets/ang3loliveira/malware-analysis-datasets-top1000-pe-imports) — regenerate with `node scripts/build-malware-hunt-samples.mjs`).

**Related:** `Threat_Hunting_Playbook.ipynb` (VPC watchlist + timestats), `Cribl_Search_Lookup_Magics.ipynb`, `Cribl_API_Examples.ipynb`.

## Prerequisites

1. Hosted Cribl app with **Cribl Search**, `%%cribl_api`, and lookup magics.
2. Permission to create **dataset providers** and **datasets** in `default_search` (or use **Fallback** below).
3. CSV mirrors ship **inside the app pack** at `{CRIBL_BASE_PATH}/data/malware-hunt/` (preferred). GitHub `raw` URLs work only after those files are on `main`.
4. **No pack proxy / `proxies.yml` changes** — Search fetches URLs via provider / `externaldata`.
4. Lookups ≤ **10,000** rows. Heavy cells use `timeout=300` (seconds).

In [ ]:
# HTTPS URLs for Cribl Search (provider + externaldata). Prefer CSVs bundled in this app pack.
from js import window

GITHUB_MIRROR = (
    "https://raw.githubusercontent.com/michaelhyatt/notebook-app/main/public/data/malware-hunt"
)


def _pack_data_url(name: str) -> str | None:
    try:
        base = (window.CRIBL_BASE_PATH or "").strip()
        origin = str(window.location.origin or "")
        if not base or not origin or origin == "null":
            return None
        if not base.startswith("/"):
            base = "/" + base
        return f"{origin.rstrip('/')}{base.rstrip('/')}/data/malware-hunt/{name}"
    except Exception:
        return None


PE_CSV_URL = _pack_data_url("pe_imports_hunt.csv") or f"{GITHUB_MIRROR}/pe_imports_hunt.csv"
TI_CSV_URL = _pack_data_url("malwarebazaar_ti_lookup.csv") or (
    f"{GITHUB_MIRROR}/malwarebazaar_ti_lookup.csv"
)
print("PE (Search will HTTP GET):", PE_CSV_URL)
print("TI (Search externaldata):", TI_CSV_URL)

## A) Create Generic HTTP provider (PE imports)

Search performs the HTTP GET when you query the federated dataset. If this cell returns **409/403**, skip to **Fallback** at the end.

In [ ]:
%%cribl_api POST /m/default_search/search/dataset-providers var=pe_provider_res response=json template=on
json:
  type: api_http
  id: notebook_pe_http
  description: PE import features (malware hash hunt example)
  authenticationMethod: none
  availableEndpoints:
    - name: pe_imports
      method: GET
      url: "{{ PE_CSV_URL }}"
      headers: []
      dataField: ""

## B) Create federated dataset

In [ ]:
%%cribl_api POST /m/default_search/search/datasets var=pe_dataset_res response=json
json:
  type: api_http
  id: notebook_pe_imports
  description: PE imports sample for hash threat hunt
  provider: notebook_pe_http
  enabledEndpoints:
    - pe_imports
  filter: "true"
  searchVersion: v1
  breakerRulesets:
    - Cribl Search
  metadata:
    enableAcceleration: false

## C) Probe PE CSV via Search (`externaldata`)

Validates the URL **before** the HTTP dataset provider runs. If this fails, fix `PE_CSV_URL` (re-run the cell above on your hosted app pack).

In [ ]:
%%cribl_search var=pe_probe lang=kql limit=5 preview=true earliest=-1h latest=now timeout=300 template=on
externaldata
[
  "{{ PE_CSV_URL }}"
]
with(
  datatype="CSV Datatypes"
)
| limit 5

## D) Preview federated dataset

Requires §A–B. On failure, read the **failed** message (job id, query, Search error text). Use **§D-alt** if the provider is blocked.

In [ ]:
%%cribl_search var=main_preview lang=kql limit=50 preview=true earliest=-1h latest=now timeout=300
dataset="notebook_pe_imports"
| limit 10

## D-alt) Preview PE via `externaldata` only (no provider)

Run this if §D fails but §C succeeded. Sets `USE_PE_EXTERNALDATA = True` for the hunt in §G-alt.

In [ ]:
USE_PE_EXTERNALDATA = True


In [ ]:
%%cribl_search var=main_preview lang=kql limit=50 preview=true earliest=-1h latest=now timeout=300 template=on
externaldata
[
  "{{ PE_CSV_URL }}"
]
with(
  datatype="CSV Datatypes"
)
| project hash, GetProcAddress, VirtualAlloc, WriteProcessMemory, URLDownloadToFileA
| limit 50

## E) Load MalwareBazaar TI via `externaldata` (Search fetch)

Same pattern as `Anomaly_Detection_PyOD.ipynb` — retrieval happens in Search, not in Pyodide.

In [ ]:
%%cribl_search var=mb_ti_raw lang=kql limit=0 preview=true earliest=-1h latest=now timeout=300 template=on
externaldata
[
  "{{ TI_CSV_URL }}"
]
with(
  datatype="CSV Datatypes"
)

## F) Normalize MD5 and publish lookup

In [ ]:
import pandas as pd


def _pick_col(frame, *names):
    keys = {str(c).strip().lower(): c for c in frame.columns}
    for n in names:
        if n in keys:
            return keys[n]
    return None


md5_col = _pick_col(mb_ti_raw, "md5", "md5_hash", "hash")
if md5_col is None:
    raise ValueError(f"Expected md5 column from Search; got {list(mb_ti_raw.columns)}")

mb_lookup_df = pd.DataFrame()
mb_lookup_df["md5"] = mb_ti_raw[md5_col].astype(str).str.strip().str.lower()
for col in ("signature", "tags", "malware_family", "file_type", "first_seen"):
    src = _pick_col(mb_ti_raw, col)
    if src:
        mb_lookup_df[col] = mb_ti_raw[src]

mb_lookup_df = mb_lookup_df.dropna(subset=["md5"])
mb_lookup_df = mb_lookup_df[mb_lookup_df["md5"].str.len() == 32]
mb_lookup_df = mb_lookup_df.drop_duplicates(subset=["md5"]).head(10000)

print("lookup rows:", len(mb_lookup_df))
mb_lookup_df.head(8)

In [ ]:
%%cribl_save_search_lookup notebook_mb_ti.csv var=mb_lookup_df replace=true

In [ ]:
%%cribl_search var=lookup_preview lang=kql limit=10 preview=true earliest=-1h latest=now
dataset="$vt_lookups" lookupFile="notebook_mb_ti"
| limit 10

## G) Hunt — join PE dataset to TI lookup on MD5

Do not start with `let` — `%%cribl_search` prepends `cribl` only when the body begins with `dataset=`.

In [ ]:
%%cribl_search var=hunt_hits lang=kql limit=5000 preview=true earliest=-1h latest=now timeout=300
dataset="notebook_pe_imports"
| extend hash = tolower(tostring(hash))
| join kind=inner (
    dataset="$vt_lookups" lookupFile="notebook_mb_ti"
  ) on $left.hash == $right.md5
| project hash, malware_family, signature, tags, GetProcAddress, VirtualAlloc, WriteProcessMemory, CreateRemoteThread, URLDownloadToFileA
| limit 2000

## G-alt) Hunt with `externaldata` + join (no federated dataset)

Use when §D failed but §D-alt succeeded (`USE_PE_EXTERNALDATA = True`).

In [ ]:
%%cribl_search var=hunt_hits lang=kql limit=5000 preview=true earliest=-1h latest=now timeout=300 template=on
let pe =
  externaldata
  [
    "{{ PE_CSV_URL }}"
  ]
  with(
    datatype="CSV Datatypes"
  )
  | extend hash = tolower(tostring(hash));
pe
| join kind=inner (
    dataset="$vt_lookups" lookupFile="notebook_mb_ti"
  ) on $left.hash == $right.md5
| project hash, malware_family, signature, tags, GetProcAddress, VirtualAlloc, WriteProcessMemory, CreateRemoteThread, URLDownloadToFileA
| limit 2000

## H) Checkpoint — match rate

In [ ]:
%%cribl_search var=pe_total lang=kql limit=0 preview=false earliest=-1h latest=now timeout=300
dataset="notebook_pe_imports"
| summarize pe_samples = dcount(hash)

In [ ]:
import pandas as pd

hits = len(hunt_hits)
use_ext = globals().get("USE_PE_EXTERNALDATA", False)
if use_ext:
    pe_n = main_preview["hash"].nunique() if "hash" in main_preview.columns else len(main_preview)
    print("(checkpoint uses §D-alt / externaldata PE preview)")
else:
    pe_n = int(pe_total.iloc[0]["pe_samples"]) if len(pe_total) else 0
rate = (hits / pe_n * 100) if pe_n else 0
print(f"Join hits: {hits} / distinct PE hashes: {pe_n} ({rate:.1f}%)")
if hits == 0:
    raise ValueError("No join hits — verify PE source, lookup, and MD5 casing.")
hunt_hits.head(10)

## I) Visualize TI families and suspicious imports

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

SUSPICIOUS = [
    "VirtualAlloc",
    "WriteProcessMemory",
    "CreateRemoteThread",
    "URLDownloadToFileA",
    "WinExec",
    "IsDebuggerPresent",
]

fam_col = next((c for c in hunt_hits.columns if str(c).lower() == "malware_family"), None)
if fam_col is None:
    raise ValueError(f"Expected malware_family in hunt_hits; got {list(hunt_hits.columns)}")

by_family = hunt_hits[fam_col].value_counts().head(8)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
by_family.sort_values().plot(kind="barh", ax=axes[0], color="#c44e52")
axes[0].set_title("Hunt hits by malware family")
axes[0].set_xlabel("Samples")

present = [c for c in SUSPICIOUS if c in hunt_hits.columns]
if present:
    means = hunt_hits[present].apply(pd.to_numeric, errors="coerce").fillna(0).mean().sort_values(ascending=False)
    means.plot(kind="bar", ax=axes[1], color="#4c72b0")
    axes[1].set_title("Mean import flag rate (matched samples)")
    axes[1].set_ylabel("Fraction with flag = 1")
    axes[1].tick_params(axis="x", rotation=45)
else:
    axes[1].text(0.5, 0.5, "Import columns not in projection", ha="center")

plt.tight_layout()
plt.show()
print("Interpretation: inner join surfaces PE rows with both import features and TI labels on the same MD5.")

## Interpretation

- **Inner join** answers: *which PE import profiles in our sample also have MalwareBazaar-style TI on the same hash?*
- **Match rate** below 100% is expected: not every PE row has TI coverage; hash format drift lowers matches.
- **MD5 reuse** — one hash can map to multiple submissions over time; treat TI as contextual, not ground truth.
- **Production** — replace curated GitHub CSVs with your own provider endpoints and authenticated TI feeds (still via Search, not Pyodide proxy).

## Cleanup

In [ ]:
%%cribl_delete_search_lookup notebook_mb_ti.csv

In [ ]:
%%cribl_api DELETE /m/default_search/search/datasets/notebook_pe_imports var=_ds_del response=json

In [ ]:
%%cribl_api DELETE /m/default_search/search/dataset-providers/notebook_pe_http var=_prov_del response=json

## Fallback (Search-only, no dataset provider)

If **§A–B** fail (permissions or provider already exists), load **both** CSVs with **`externaldata`** — still Cribl Search, not pack proxy:

1. Run the URL cell, then a `%%cribl_search` `externaldata` cell for `PE_CSV_URL` → `pe_df`.
2. Re-run **§D–E** for the TI lookup.
3. In Python, `inner` merge on `hash`/`md5` for charts (KQL join requires the federated dataset on the left).

Ensure CSV URLs are reachable from your Search workers (merge to `main` on GitHub or host copies your tenant can reach).